In [0]:
PATH_FEATURES = "/Volumes/workspace/default/bronze/silver/sensor_features/"
PATH_GOLD     = "/Volumes/workspace/default/gold/"

print("✅ Chemins définis")

✅ Chemins définis


In [0]:
    from pyspark.sql import functions as F
from pyspark.sql.types import *

print("✅ Imports OK")

✅ Imports OK


In [0]:
df = spark.read.parquet(PATH_FEATURES)
print(f"✅ Silver chargé : {df.count()} lignes × {len(df.columns)} colonnes")

✅ Silver chargé : 10000 lignes × 26 colonnes


In [0]:
# Voir tous les volumes disponibles
display(spark.sql("SHOW VOLUMES IN workspace.default"))

database,volume_name
default,bronze
default,features
default,silver


In [0]:
%sql
-- Dans une cellule SQL (ou avec spark.sql)
CREATE VOLUME IF NOT EXISTS workspace.default.gold

In [0]:
# Étape 2 : créer le volume gold
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.gold")
print("✅ Volume gold créé")

# Étape 3 : vérifier
PATH_GOLD = "/Volumes/workspace/default/gold/"
dbutils.fs.mkdirs(PATH_GOLD)
print(f"✅ Chemin Gold accessible : {PATH_GOLD}")

✅ Volume gold créé
✅ Chemin Gold accessible : /Volumes/workspace/default/gold/


In [0]:
# ============================================================
# CELLULE 3 — Création DIM_DATE
# ============================================================
# Pourquoi ? Power BI a besoin d'une table calendrier séparée
# pour filtrer par mois, semaine, trimestre facilement

dim_date = df.select("timestamp").distinct() \
    .withColumn("date_id",     F.date_format("timestamp", "yyyyMMdd").cast("int")) \
    .withColumn("full_date",   F.to_date("timestamp")) \
    .withColumn("year",        F.year("timestamp")) \
    .withColumn("month",       F.month("timestamp")) \
    .withColumn("month_name",  F.date_format("timestamp", "MMMM")) \
    .withColumn("week",        F.weekofyear("timestamp")) \
    .withColumn("day_of_week", F.dayofweek("timestamp")) \
    .withColumn("is_weekend",  F.when(
                                F.dayofweek("timestamp").isin([1, 7]), 1
                               ).otherwise(0)) \
    .drop("timestamp") \
    .distinct()

dim_date.write.mode("overwrite").parquet(PATH_GOLD + "dim_date/")
dim_date = spark.read.parquet(PATH_GOLD + "dim_date/")
print(f"✅ DIM_DATE créée : {dim_date.count()} dates distinctes")

✅ DIM_DATE créée : 7 dates distinctes


In [0]:
# ============================================================
# CELLULE 4 — Création DIM_STATION
# ============================================================
# station_type existe dans Silver (L / M / H)
# On enrichit avec des informations contextuelles OCP

dim_station = df.select("station_type").distinct() \
    .withColumn("station_id",   F.monotonically_increasing_id()) \
    .withColumn("station_name", F.when(F.col("station_type") == "L", "Station Légère")
                                 .when(F.col("station_type") == "M", "Station Moyenne")
                                 .otherwise("Station Lourde")) \
    .withColumn("region",       F.when(F.col("station_type") == "L", "Khouribga")
                                 .when(F.col("station_type") == "M", "Béni Mellal")
                                 .otherwise("Jorf Lasfar"))

dim_station.write.mode("overwrite").parquet(PATH_GOLD + "dim_station/")
dim_station = spark.read.parquet(PATH_GOLD + "dim_station/")
print(f"✅ DIM_STATION créée : {dim_station.count()} stations")
dim_station.show()

✅ DIM_STATION créée : 3 stations
+------------+----------+---------------+-----------+
|station_type|station_id|   station_name|     region|
+------------+----------+---------------+-----------+
|           L|         0| Station Légère|  Khouribga|
|           M|         1|Station Moyenne|Béni Mellal|
|           H|         2| Station Lourde|Jorf Lasfar|
+------------+----------+---------------+-----------+



In [0]:
# ============================================================
# CELLULE 5 — Création DIM_EQUIPMENT
# ============================================================

equipment_data = [
    (1, "Pompe centrifuge",  "Sulzer",  90),
    (2, "Compresseur",       "Atlas Copco", 60),
    (3, "Vanne de régulation","Flowserve", 180),
]

schema = StructType([
    StructField("equipment_id",           IntegerType(), False),
    StructField("equipment_type",         StringType(),  False),
    StructField("manufacturer",           StringType(),  False),
    StructField("maintenance_cycle_days", IntegerType(), False),
])

dim_equipment = spark.createDataFrame(equipment_data, schema)
dim_equipment.write.mode("overwrite").parquet(PATH_GOLD + "dim_equipment/")
dim_equipment = spark.read.parquet(PATH_GOLD + "dim_equipment/")
print(f"✅ DIM_EQUIPMENT créée : {dim_equipment.count()} équipements")
dim_equipment.show()

✅ DIM_EQUIPMENT créée : 3 équipements
+------------+-------------------+------------+----------------------+
|equipment_id|     equipment_type|manufacturer|maintenance_cycle_days|
+------------+-------------------+------------+----------------------+
|           3|Vanne de régulation|   Flowserve|                   180|
|           2|        Compresseur| Atlas Copco|                    60|
|           1|   Pompe centrifuge|      Sulzer|                    90|
+------------+-------------------+------------+----------------------+



In [0]:
# ============================================================
# CELLULE 6 — Création DIM_KPI
# ============================================================
# Ce tableau définit officiellement les KPIs du dashboard OCP
# C'est la "charte des KPIs" — tout le monde utilise la même définition

kpi_data = [
    (1, "MTBF",           "Mean Time Between Failures — Durée moyenne entre 2 pannes", "heures", 720),
    (2, "Disponibilite",  "% de temps où la pompe fonctionne normalement",             "%",      98.5),
    (3, "Taux_pannes",    "Nombre de pannes / nombre total de mesures",                "%",      2.0),
    (4, "Usure_moyenne",  "Moyenne de l'usure outil sur la période",                  "min",    150),
]

schema_kpi = StructType([
    StructField("kpi_id",         IntegerType(), False),
    StructField("kpi_name",       StringType(),  False),
    StructField("kpi_definition", StringType(),  False),
    StructField("unit",           StringType(),  False),
    StructField("target_value",   DoubleType(),  False),
])

dim_kpi = spark.createDataFrame(kpi_data, schema_kpi)
dim_kpi.write.mode("overwrite").parquet(PATH_GOLD + "dim_kpi/")
dim_kpi = spark.read.parquet(PATH_GOLD + "dim_kpi/")
print(f"✅ DIM_KPI créée : {dim_kpi.count()} KPIs définis")
dim_kpi.show(truncate=False)

✅ DIM_KPI créée : 4 KPIs définis
+------+-------------+---------------------------------------------------------+------+------------+
|kpi_id|kpi_name     |kpi_definition                                           |unit  |target_value|
+------+-------------+---------------------------------------------------------+------+------------+
|1     |MTBF         |Mean Time Between Failures — Durée moyenne entre 2 pannes|heures|720.0       |
|2     |Disponibilite|% de temps où la pompe fonctionne normalement            |%     |98.5        |
|4     |Usure_moyenne|Moyenne de l'usure outil sur la période                  |min   |150.0       |
|3     |Taux_pannes  |Nombre de pannes / nombre total de mesures               |%     |2.0         |
+------+-------------+---------------------------------------------------------+------+------------+



In [0]:
# ============================================================
# CELLULE 7 — Création FACT_SENSOR_EVENTS
# ============================================================
# C'est la table centrale — elle joint toutes les dimensions
# via leurs clés (station_id, date_id, equipment_id)

# Jointure avec DIM_STATION pour récupérer station_id
df_with_station = df.join(
    dim_station.select("station_type", "station_id"),
    on="station_type",
    how="left"
)

# Création de date_id pour jointure avec DIM_DATE
df_with_date = df_with_station.withColumn(
    "date_id",
    F.date_format("timestamp", "yyyyMMdd").cast("int")
)

# Attribution equipment_id (basé sur station_type pour simuler)
df_with_equip = df_with_date.withColumn(
    "equipment_id",
    F.when(F.col("station_type") == "L", 1)
     .when(F.col("station_type") == "M", 2)
     .otherwise(3)
)

# Construction de la FACT finale
fact_sensor = df_with_equip.select(
    "sensor_id",
    "station_id",
    "equipment_id",
    "date_id",
    "timestamp",
    "air_temp_k",
    "process_temp_k",
    "rotation_speed_rpm",
    "torque_nm",
    "tool_wear_min",
    "power_estimated_w",
    "temp_diff",
    "tool_wear_rate",
    "rolling_mean_temp_1h",
    "rolling_std_temp_1h",
    "failure_type",
    F.when(F.col("failure_type") != "NONE", 1).otherwise(0).alias("is_failure")
)

fact_sensor.write.mode("overwrite").parquet(PATH_GOLD + "fact_sensor_events/")
fact_sensor = spark.read.parquet(PATH_GOLD + "fact_sensor_events/")
print(f"✅ FACT_SENSOR_EVENTS créée : {fact_sensor.count()} lignes")

✅ FACT_SENSOR_EVENTS créée : 10000 lignes


In [0]:
# ============================================================
# CELLULE 8 — Vérification Star Schema complet
# ============================================================

print("=" * 50)
print("     GOLD LAYER — STAR SCHEMA OCP")
print("=" * 50)
print(f"FACT_SENSOR_EVENTS : {fact_sensor.count():>8} lignes")
print(f"DIM_DATE           : {dim_date.count():>8} dates")
print(f"DIM_STATION        : {dim_station.count():>8} stations")
print(f"DIM_EQUIPMENT      : {dim_equipment.count():>8} équipements")
print(f"DIM_KPI            : {dim_kpi.count():>8} KPIs")
print("=" * 50)

# Vérifier les pannes dans la FACT
pannes = fact_sensor.filter(F.col("is_failure") == 1).count()
print(f"\n⚠️  Pannes enregistrées : {pannes}")
print(f"✅ Taux de disponibilité : {round((1 - pannes/fact_sensor.count())*100, 2)}%")

     GOLD LAYER — STAR SCHEMA OCP
FACT_SENSOR_EVENTS :    10000 lignes
DIM_DATE           :        7 dates
DIM_STATION        :        3 stations
DIM_EQUIPMENT      :        3 équipements
DIM_KPI            :        4 KPIs

⚠️  Pannes enregistrées : 348
✅ Taux de disponibilité : 96.52%
